# 02 — Facilities ID Map Silver

Builds `public_health.silver.facilities_id_map_silver` — the stable facility identity map that resolves NHFR and HFSRB identifiers into a single `facility_key`.

### Sources
- NHFR: `public_health.bronze.facilities_nhfr_bronze` (1,359 hospitals)
- HFSRB hospitals: `public_health.bronze.facilities_hfsrb_hospitals_bronze` (1,336 hospitals)

### Key Rules (facilities-silver-plan v1.5, section 2 + lineage contract v1.2)
- Comparison uses `nhfr_id` = numeric value of the code, never a padded string
- NHFR: `nhfr_id = cast(substr(Health_Facility_Code, 4) AS BIGINT)`
- HFSRB: Parse `^(DOH)?\d{1,15}(\.0)?$` → numeric; text tokens → null
- Duplicate-code resolution (2.4): exact dupes → keep one; same facility → keep latest; different facilities → ambiguous
- Gate: each `nhfr_id` maps to at most 1 HFSRB hospital record
- Fallback: normalized name + municipality (2.5)
- `facility_key` = uuid(), assigned once, never re-derived
- Ambiguous codes are quarantined (`quality.quarantine`, reason `ambiguous_nhfr_code`) then fall back to name match
- Lineage (plan 5.2): 6 columns — `_silver_build_id`, `_source_key`, `_source_file`, `_source_sheet`, `_source_row_num`, `_as_of_date`. `_first_build_id` replaces `first_mapped_at`/`last_seen_at`; load facts live in `quality.build_inputs` / `build_log` (via `v_build_lineage`)

### Processing Flow
Bronze → Parse nhfr_id → Resolve duplicates → Match → Assign facility_key → Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
import json
from datetime import datetime, timezone

NHFR_BRONZE = "public_health.bronze.facilities_nhfr_bronze"
HFSRB_BRONZE = "public_health.bronze.facilities_hfsrb_hospitals_bronze"
GEOGRAPHY_SILVER = "public_health.silver.geography_silver"
TARGET_TABLE = "public_health.silver.facilities_id_map_silver"
METADATA_TABLE = "public_health.bronze.ingestion_metadata"
BUILD_LOG_TABLE = "public_health.quality.build_log"
BUILD_INPUTS_TABLE = "public_health.quality.build_inputs"
ROW_DISPOSITION_TABLE = "public_health.quality.row_disposition"
QUARANTINE_TABLE = "public_health.quality.quarantine"

EXPECTED_TOTAL_ROWS = 2695

print("Config loaded")
print(f"NHFR bronze    : {NHFR_BRONZE}")
print(f"HFSRB bronze   : {HFSRB_BRONZE}")
print(f"Geography silver: {GEOGRAPHY_SILVER}")
print(f"Target table   : {TARGET_TABLE}")

In [0]:
# v1.5 build sequence, step 1: capture inputs (plan 5.5, contract 5.1).
# Inputs: NHFR CSV (no sheet) + HFSRB hospitals workbook (Government/Private sheets).
# Each input matched to exactly one processed ingestion_metadata row via
# (source_name, source_sheet, retrieved_at = _ingested_at) — contract 3.5.
SILVER_BUILD_ID = str(uuid.uuid4())
BUILD_STARTED_AT = datetime.now(timezone.utc)
print(f"Silver build ID: {SILVER_BUILD_ID}")
print(f"Build started  : {BUILD_STARTED_AT.isoformat()}")

def latest_delta_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table}").orderBy(F.desc("version")).first()["version"]

nhfr_bronze_version = latest_delta_version(NHFR_BRONZE)
hfsrb_bronze_version = latest_delta_version(HFSRB_BRONZE)
print(f"NHFR bronze version captured : {nhfr_bronze_version}")
print(f"HFSRB bronze version captured: {hfsrb_bronze_version}")

meta = spark.table(METADATA_TABLE).filter(F.col("status") == "processed")

# NHFR input: CSV source, no sheet
nhfr_loads = (
    spark.sql(f"SELECT * FROM {NHFR_BRONZE} VERSION AS OF {nhfr_bronze_version}")
    .groupBy("_ingested_at")
    .agg(F.first("_source_file", ignorenulls=True).alias("file"), F.count("*").alias("rows"))
)
nhfr_captured = nhfr_loads.join(
    meta.filter(F.col("source_name") == "doh_nhfr")
        .select(F.col("retrieved_at").alias("meta_ts"), F.col("batch_id"), F.col("content_checksum")),
    F.col("_ingested_at") == F.col("meta_ts"), "left",
).collect()

# HFSRB inputs: one workbook, Government + Private sheets
hfsrb_loads = (
    spark.sql(f"SELECT * FROM {HFSRB_BRONZE} VERSION AS OF {hfsrb_bronze_version}")
    .groupBy("_source_sheet", "_ingested_at")
    .agg(F.first("_source_file", ignorenulls=True).alias("file"), F.count("*").alias("rows"))
)
hfsrb_captured = hfsrb_loads.join(
    meta.filter(F.col("source_name") == "doh_hfsrb_hospitals")
        .select(F.col("source_sheet").alias("meta_sheet"), F.col("retrieved_at").alias("meta_ts"),
                F.col("batch_id"), F.col("content_checksum")),
    (F.col("_source_sheet") == F.col("meta_sheet")) & (F.col("_ingested_at") == F.col("meta_ts")),
    "left",
).orderBy("_source_sheet").collect()

# Gates: exactly 1 NHFR input + 2 HFSRB sheets, each with exactly one metadata match
assert len(nhfr_captured) == 1, f"Expected 1 NHFR input, got {len(nhfr_captured)}"
assert len(hfsrb_captured) == 2, f"Expected 2 HFSRB sheets (Government/Private), got {len(hfsrb_captured)}"
for r in list(nhfr_captured) + list(hfsrb_captured):
    assert r["batch_id"] is not None, "An input has no exactly-one processed ingestion_metadata match"

for r in nhfr_captured:
    print(f"  NHFR: {r['file']} rows={r['rows']} batch={r['batch_id']}")
for r in hfsrb_captured:
    print(f"  HFSRB {r['_source_sheet']}: {r['file']} rows={r['rows']} batch={r['batch_id']}")

captured_inputs_full = (
    [(NHFR_BRONZE, nhfr_bronze_version, "doh_nhfr", r["file"], None,
      r["batch_id"], r["content_checksum"], r["_ingested_at"], r["rows"])
     for r in nhfr_captured]
    + [(HFSRB_BRONZE, hfsrb_bronze_version, "doh_hfsrb_hospitals", r["file"], r["_source_sheet"],
       r["batch_id"], r["content_checksum"], r["_ingested_at"], r["rows"])
       for r in hfsrb_captured]
)
total_input_rows = sum(r[8] for r in captured_inputs_full)
print(f"\nTotal input rows: {total_input_rows} (expected {EXPECTED_TOTAL_ROWS})")
assert total_input_rows == EXPECTED_TOTAL_ROWS
print("Input capture gates passed")

In [0]:
# NHFR: Health_Facility_Code is always DOH + 15 digits
# nhfr_id = cast(substr(code, 4) AS BIGINT) — all 1,359 rows
nhfr_df = (
    spark.sql(f"SELECT * FROM {NHFR_BRONZE} VERSION AS OF {nhfr_bronze_version}")
    .select(
        F.lit("doh_nhfr").alias("_source_key"),
        F.col("Health_Facility_Code").alias("source_id_raw"),
        F.col("Health_Facility_Code_Short").alias("source_id_short"),
        F.col("Facility_Name").alias("facility_name"),
        F.col("City/Municipality_Name").alias("municipality_name"),
        F.col("City/Municipality_PSGC").alias("municipality_psgc"),
        F.col("Province_Name").alias("province_name"),
        F.col("Region_Name").alias("region_name"),
        F.regexp_extract(F.col("Health_Facility_Code"), r"^DOH(\d{15})$", 1)
            .cast(LongType()).alias("nhfr_id"),
        F.col("_source_file").alias("source_file"),
        F.col("_source_row_num").alias("source_row_num"),
        F.col("_as_of_date").alias("as_of_date"),
    )
)

nhfr_count = nhfr_df.count()
nhfr_with_id = nhfr_df.filter(F.col("nhfr_id").isNotNull()).count()
nhfr_unique = nhfr_df.filter(F.col("nhfr_id").isNotNull()).select("nhfr_id").distinct().count()

print(f"NHFR rows: {nhfr_count}")
print(f"NHFR with valid nhfr_id: {nhfr_with_id}")
print(f"NHFR unique nhfr_ids: {nhfr_unique}")
assert nhfr_with_id == nhfr_count, f"Expected all {nhfr_count} NHFR rows to have valid nhfr_id"
assert nhfr_unique == nhfr_count, f"Expected {nhfr_count} unique nhfr_ids, got {nhfr_unique}"
print("NHFR validation passed: all rows have unique nhfr_id")

In [0]:
# HFSRB: nhfr_code has various formats (3-5 digits, DOH+15d, DOH+5d, null, text tokens)
# Valid: ^(DOH)?\d{1,15}(\.0)?$ → extract numeric part → cast to BIGINT
# Text tokens (none, duplicate NHFR, NO NHFR) → null with reason invalid_code_token

nhfr_code_clean = F.trim(F.col("nhfr_code"))
is_valid = (
    F.col("nhfr_code").isNotNull()
    & (nhfr_code_clean != "")
    & nhfr_code_clean.rlike(r"^(DOH)?\d{1,15}(\.0)?$")
)
numeric_part = F.regexp_replace(nhfr_code_clean, r"^(DOH)?(\d{1,15})(\.0)?$", "$2")

hfsrb_df = (
    spark.sql(f"SELECT * FROM {HFSRB_BRONZE} VERSION AS OF {hfsrb_bronze_version}")
    .select(
        F.lit("doh_hfsrb_hospitals").alias("_source_key"),
        nhfr_code_clean.alias("source_id_raw"),
        F.col("facility_name").alias("facility_name"),
        F.col("license_no").alias("license_no"),
        F.col("city_municipality").alias("municipality_name"),
        F.col("province").alias("province_name"),
        F.col("region").alias("region_name"),
        F.col("ownership").alias("ownership"),
        F.col("abc").alias("abc"),
        F.col("validity_from").alias("validity_from"),
        F.col("validity_until").alias("validity_until"),
        F.col("_source_file").alias("source_file"),
        F.col("_source_sheet").alias("source_sheet"),
        F.col("_source_row_num").alias("source_row_num"),
        F.col("_as_of_date").alias("as_of_date"),
        F.when(is_valid, numeric_part.cast(LongType())).otherwise(F.lit(None)).alias("nhfr_id"),
        is_valid.alias("code_validated"),
        F.when(
            (~is_valid) & F.col("nhfr_code").isNotNull() & (nhfr_code_clean != ""),
            F.lit("invalid_code_token")
        ).otherwise(F.lit(None)).alias("invalid_reason"),
    )
)

hfsrb_count = hfsrb_df.count()
hfsrb_with_id = hfsrb_df.filter(F.col("nhfr_id").isNotNull()).count()
hfsrb_null = hfsrb_df.filter(F.col("nhfr_id").isNull()).count()
hfsrb_unique = hfsrb_df.filter(F.col("nhfr_id").isNotNull()).select("nhfr_id").distinct().count()

print(f"HFSRB hospital rows: {hfsrb_count}")
print(f"HFSRB with valid nhfr_id: {hfsrb_with_id}")
print(f"HFSRB null/invalid nhfr_id: {hfsrb_null}")
print(f"HFSRB unique nhfr_ids: {hfsrb_unique}")

# Show invalid code tokens
print("\nInvalid code tokens:")
hfsrb_df.filter(F.col("invalid_reason").isNotNull()) \
    .select("source_id_raw", "facility_name", "municipality_name") \
    .show(20, truncate=False)

In [0]:
# Section 2.4: For HFSRB rows sharing one nhfr_id
# Find nhfr_ids shared by multiple HFSRB rows
duplicate_ids = (
    hfsrb_df
    .filter(F.col("nhfr_id").isNotNull())
    .groupBy("nhfr_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .orderBy(F.desc("row_count"))
)

dup_count = duplicate_ids.count()
if dup_count > 0:
    dup_rows = duplicate_ids.agg(F.sum("row_count").alias("total")).collect()[0]["total"]
    print(f"NHFR ids hit by more than one HFSRB row: {dup_count}")
    print(f"Total HFSRB rows in duplicate groups: {dup_rows}")
    
    # Show the duplicate groups
    dup_ids_list = [row["nhfr_id"] for row in duplicate_ids.collect()]
    duplicate_details = (
        hfsrb_df
        .filter(F.col("nhfr_id").isin(dup_ids_list))
        .select("nhfr_id", "facility_name", "municipality_name", "province_name",
               "license_no", "validity_until", "source_file", "source_row_num")
        .orderBy("nhfr_id")
    )
    display(duplicate_details)
else:
    print("No duplicate nhfr_ids found in HFSRB")

In [0]:
# Classify each duplicate group into type 1/2/3 and resolve
# Type 1: exact duplicates (all business columns equal) → keep one, others → duplicate_of
# Type 2: same facility (same license_no or same normalized name + municipality) → keep latest validity_until
# Type 3: different facilities → ambiguous, code not used, fall back to name+municipality

import re

dup_ids_list = [row["nhfr_id"] for row in duplicate_ids.collect()] if dup_count > 0 else []

resolution_rows = []
for nhfr_id_val in dup_ids_list:
    group = (
        hfsrb_df
        .filter(F.col("nhfr_id") == nhfr_id_val)
        .select("facility_name", "municipality_name", "province_name",
               "license_no", "validity_until", "source_file", "source_row_num")
        .collect()
    )
    
    # Check type 1: exact duplicates (same name + municipality + license_no)
    signatures = set()
    for r in group:
        sig = (r["facility_name"], r["municipality_name"], r["license_no"])
        signatures.add(sig)
    
    if len(signatures) == 1:
        res_type = "exact_duplicate"
    else:
        # Check type 2: same facility (same normalized name + municipality)
        norm_sigs = set()
        for r in group:
            norm_name = re.sub(r'[^A-Z0-9]', '', (r["facility_name"] or "").upper())
            norm_muni = (r["municipality_name"] or "").upper().strip()
            norm_sigs.add((norm_name, norm_muni))
        
        if len(norm_sigs) == 1:
            res_type = "same_facility"
        else:
            res_type = "ambiguous"
            names = [(r["facility_name"], r["municipality_name"]) for r in group]
            print(f"AMBIGUOUS nhfr_id={nhfr_id_val}: {names}")
    
    resolution_rows.append({
        "nhfr_id": nhfr_id_val,
        "resolution_type": res_type,
        "row_count": len(group),
    })

if resolution_rows:
    resolution_df = spark.createDataFrame(resolution_rows)
    print("\nDuplicate resolution summary:")
    resolution_df.groupBy("resolution_type").agg(
        F.count("*").alias("groups"),
        F.sum("row_count").alias("rows")
    ).show()
else:
    print("No duplicates to resolve")
    resolution_df = spark.createDataFrame(
        [Row(nhfr_id=0, resolution_type="none", row_count=0)],
        schema="nhfr_id long, resolution_type string, row_count int"
    )

In [0]:
# Build the resolved HFSRB DataFrame:
# - Type 1 (exact_duplicate): keep one row, others get disposition='duplicate_of'
# - Type 2 (same_facility): keep latest validity_until, others get disposition='duplicate_of'
# - Type 3 (ambiguous): all rows get disposition='ambiguous_nhfr_code', nhfr_id set to null (fall back to name match)

if dup_count > 0:
    # Create a lookup of nhfr_id → resolution_type
    res_type_map = {r["nhfr_id"]: r["resolution_type"] for r in resolution_rows}
    
    # Broadcast the resolution type to each HFSRB row
    res_type_df = spark.createDataFrame(
        [(k, v) for k, v in res_type_map.items()],
        schema="nhfr_id long, dup_resolution_type string"
    )
    
    hfsrb_resolved = (
        hfsrb_df
        .join(res_type_df, on="nhfr_id", how="left")
        .withColumn(
            "disposition",
            F.when(F.col("dup_resolution_type").isNull(), F.lit("kept"))
            .otherwise(F.col("dup_resolution_type"))
        )
        # For ambiguous: set nhfr_id to null (code not used)
        .withColumn(
            "nhfr_id_resolved",
            F.when(F.col("dup_resolution_type") == "ambiguous", F.lit(None))
            .otherwise(F.col("nhfr_id"))
        )
    )
    
    # For type 1 and 2: keep one row per nhfr_id (the one to use for matching)
    # Type 1: keep the first row (arbitrary, they're identical)
    # Type 2: keep the row with the latest validity_until
    # Mark non-kept rows as duplicate_of
    from pyspark.sql.window import Window
    
    dup_window = Window.partitionBy("nhfr_id").orderBy(
        F.col("validity_until").desc_nulls_last(),
        F.col("source_row_num").asc()
    )
    
    hfsrb_resolved = (
        hfsrb_resolved
        .withColumn("dup_rank", F.row_number().over(dup_window))
        .withColumn(
            "disposition",
            F.when(
                (F.col("dup_resolution_type").isin(["exact_duplicate", "same_facility"]))
                & (F.col("dup_rank") > 1),
                F.lit("duplicate_of")
            ).otherwise(F.col("disposition"))
        )
        .withColumn(
            "nhfr_id_final",
            F.when(F.col("disposition") == "duplicate_of", F.lit(None))
            .otherwise(F.col("nhfr_id_resolved"))
        )
        .drop("dup_rank", "nhfr_id_resolved")
    )
else:
    hfsrb_resolved = hfsrb_df.withColumn("disposition", F.lit("kept")).withColumn("nhfr_id_final", F.col("nhfr_id"))

# GATE: after resolution, each nhfr_id maps to at most 1 HFSRB hospital record
gate_violations = (
    hfsrb_resolved
    .filter(F.col("nhfr_id_final").isNotNull() & (F.col("disposition") != "duplicate_of"))
    .groupBy("nhfr_id_final")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .count()
)

print(f"Gate violations (nhfr_id maps to >1 HFSRB record): {gate_violations}")
assert gate_violations == 0, f"GATE FAILED: {gate_violations} nhfr_ids still map to multiple HFSRB records"
print("Gate passed: each nhfr_id maps to at most 1 HFSRB hospital record")

# Summary
disposition_counts = hfsrb_resolved.groupBy("disposition").agg(F.count("*").alias("rows"))
print("\nDisposition summary:")
disposition_counts.show()

In [0]:
# Match HFSRB (resolved) to NHFR on nhfr_id
# Then fallback match for unmatched rows (section 2.5)

def normalize_name(name_col):
    """Normalize facility name: upper case, punctuation removed, standard abbreviations."""
    return F.upper(
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(F.trim(name_col), r"HOSP\\.", "HOSPITAL"),
                r"MED\\.", "MEDICAL"
            ),
            r"[^A-Z0-9 ]", ""
        )
    )

# NHFR lookup for name matching
from pyspark.sql.window import Window

# Deduplicate: keep only one NHFR row per (norm_name, norm_municipality)
# If multiple NHFR facilities share the same name + municipality, only the first is used
name_window = Window.partitionBy("norm_name", "norm_municipality").orderBy("nhfr_id")

nhfr_name_lookup = (
    nhfr_df
    .withColumn("norm_name", normalize_name(F.col("facility_name")))
    .withColumn("norm_municipality", F.upper(F.trim(F.col("municipality_name"))))
    .withColumn("name_rank", F.row_number().over(name_window))
    .filter(F.col("name_rank") == 1)
    .select(
        F.col("norm_name").alias("nl_norm_name"),
        F.col("norm_municipality").alias("nl_norm_municipality"),
        F.col("nhfr_id").alias("nhfr_id_by_name"),
        F.col("facility_name").alias("nhfr_matched_name"),
    )
)

# Join HFSRB to NHFR on nhfr_id, then fallback on normalized name + municipality
hfsrb_matched = (
    hfsrb_resolved
    .withColumn("norm_name", normalize_name(F.col("facility_name")))
    .withColumn("norm_municipality", F.upper(F.trim(F.col("municipality_name"))))
    # Left join on nhfr_id_final
    .join(
        nhfr_df.select(F.col("nhfr_id").alias("nhfr_match_id")),
        F.col("nhfr_id_final") == F.col("nhfr_match_id"),
        how="left"
    )
    # Left join on normalized name + municipality (for fallback)
    .join(
        nhfr_name_lookup,
        (F.col("norm_name") == F.col("nl_norm_name"))
        & (F.col("norm_municipality") == F.col("nl_norm_municipality")),
        how="left"
    )
    .withColumn(
        "match_method",
        F.when(F.col("disposition") == "duplicate_of", F.lit(None))
        .when(F.col("nhfr_match_id").isNotNull(), F.lit("nhfr_id"))
        .when(F.col("nhfr_id_by_name").isNotNull(), F.lit("name_municipality"))
        .otherwise(F.lit("new"))
    )
    .withColumn(
        "match_status",
        F.when(F.col("disposition") == "duplicate_of", F.lit("duplicate_of"))
        .when(F.col("nhfr_match_id").isNotNull() | F.col("nhfr_id_by_name").isNotNull(), F.lit("matched"))
        .when(F.col("disposition") == "ambiguous", F.lit("ambiguous"))
        .otherwise(F.lit("new_facility"))
    )
    .withColumn(
        "match_nhfr_id",
        F.coalesce(F.col("nhfr_match_id"), F.col("nhfr_id_by_name"))
    )
)

code_matched = hfsrb_matched.filter(F.col("match_method") == "nhfr_id").count()
name_matched = hfsrb_matched.filter(F.col("match_method") == "name_municipality").count()
new_fac = hfsrb_matched.filter(F.col("match_method") == "new").count()
dup_of = hfsrb_matched.filter(F.col("disposition") == "duplicate_of").count()

print(f"HFSRB matched by nhfr_id: {code_matched}")
print(f"HFSRB matched by name+municipality: {name_matched}")
print(f"HFSRB new facilities: {new_fac}")
print(f"HFSRB duplicate_of: {dup_of}")

In [0]:
# Build the unified id map
# NHFR rows: all get new facility_key, match_method='new', match_status='new_facility'
# HFSRB matched rows: use the NHFR facility_key
# HFSRB unmatched rows: get new facility_key
# HFSRB duplicate_of rows: get new facility_key (marked with disposition)

# Step 1: Generate facility_keys for all NHFR rows in Python (deterministic)
# uuid() expression is non-deterministic and re-evaluated on each materialization on serverless
nhfr_ids = [r["nhfr_id"] for r in nhfr_df.select("nhfr_id").collect()]
nhfr_keys = {nid: str(uuid.uuid4()) for nid in nhfr_ids}
nhfr_key_rows = [(k, v) for k, v in nhfr_keys.items()]
nhfr_key_df = spark.createDataFrame(nhfr_key_rows, schema="nhfr_id long, facility_key string")

# Step 2: NHFR id map — join to assign facility_keys
nhfr_id_map = (
    nhfr_df
    .join(nhfr_key_df, on="nhfr_id", how="left")
    .select(
        F.lit("doh_nhfr").alias("_source_key"),
        F.col("nhfr_id").cast("string").alias("source_record_id"),
        F.col("source_id_raw"),
        F.col("source_id_short").alias("source_id_norm"),
        F.lit(True).alias("code_validated"),
        F.col("facility_key"),
        F.lit("new").alias("match_method"),
        F.lit("new_facility").alias("match_status"),
        F.col("facility_name"),
        F.col("municipality_name"),
        F.col("nhfr_id"),
        F.lit("kept").alias("disposition"),
        F.lit(SILVER_BUILD_ID).alias("_first_build_id"),
        F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
        F.col("source_file").alias("_source_file"),
        F.lit(None).cast("string").alias("_source_sheet"),
        F.col("source_row_num").alias("_source_row_num"),
        F.col("as_of_date").alias("_as_of_date"),
    )
)

# Step 3: facility_key lookup for HFSRB matched rows (same nhfr_key_df)
facility_key_lookup = nhfr_key_df.select(
    F.col("nhfr_id").alias("fk_lookup_nhfr_id"),
    F.col("facility_key").alias("fk_lookup_key"),
)

# Step 4: Generate facility_keys for unmatched HFSRB rows in Python
hfsrb_records = (
    hfsrb_matched
    .filter(F.col("disposition") != "duplicate_of")
    .select(
        F.coalesce(
            F.col("license_no"),
            F.concat_ws("_", F.col("source_file"), F.col("source_row_num").cast("string"))
        ).alias("source_record_id"),
        F.col("match_nhfr_id"),
    )
    .collect()
)

hfsrb_new_keys = {}
for r in hfsrb_records:
    if r["match_nhfr_id"] is None:
        hfsrb_new_keys[r["source_record_id"]] = str(uuid.uuid4())

hfsrb_key_df = spark.createDataFrame(
    [(k, v) for k, v in hfsrb_new_keys.items()],
    schema="hfsrb_record_id string, hfsrb_facility_key string"
)

# Step 5: HFSRB id map — assign facility_keys from lookup or new
hfsrb_id_map = (
    hfsrb_matched
    .filter(F.col("disposition") != "duplicate_of")
    .withColumn(
        "source_record_id",
        F.coalesce(
            F.col("license_no"),
            F.concat_ws("_", F.col("source_file"), F.col("source_row_num").cast("string"))
        )
    )
    .join(
        facility_key_lookup,
        F.col("match_nhfr_id") == F.col("fk_lookup_nhfr_id"),
        how="left"
    )
    .join(
        hfsrb_key_df,
        F.col("source_record_id") == F.col("hfsrb_record_id"),
        how="left"
    )
    .withColumn(
        "facility_key",
        F.coalesce(F.col("fk_lookup_key"), F.col("hfsrb_facility_key"))
    )
    .select(
        F.lit("doh_hfsrb_hospitals").alias("_source_key"),
        F.col("source_record_id"),
        F.col("source_id_raw"),
        F.col("source_id_raw").alias("source_id_norm"),
        F.col("code_validated"),
        F.col("facility_key"),
        F.col("match_method"),
        F.col("match_status"),
        F.col("facility_name"),
        F.col("municipality_name"),
        F.col("nhfr_id_final").alias("nhfr_id"),
        F.col("disposition"),
        F.lit(SILVER_BUILD_ID).alias("_first_build_id"),
        F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
        F.col("source_file").alias("_source_file"),
        F.col("source_sheet").alias("_source_sheet"),
        F.col("source_row_num").alias("_source_row_num"),
        F.col("as_of_date").alias("_as_of_date"),
    )
)

# Step 6: Combine NHFR + HFSRB
id_map_df = nhfr_id_map.unionByName(hfsrb_id_map)

total_rows = id_map_df.count()
unique_keys = id_map_df.select("facility_key").distinct().count()
nhfr_rows = id_map_df.filter(F.col("_source_key") == "doh_nhfr").count()
hfsrb_rows = id_map_df.filter(F.col("_source_key") == "doh_hfsrb_hospitals").count()
matched_rows = id_map_df.filter(F.col("match_status") == "matched").count()

print(f"Total id map rows: {total_rows}")
print(f"  NHFR rows: {nhfr_rows}")
print(f"  HFSRB rows: {hfsrb_rows}")
print(f"Unique facility_keys: {unique_keys}")
print(f"Matched HFSRB rows (share a key with NHFR): {matched_rows}")
print(f"Expected unique keys: {total_rows - matched_rows} (total - matched sharing)")

In [0]:
# v1.5 target schema (plan 2.6, 5.2): address lineage (6 cols) + _first_build_id.
# This build re-reads every input row, so the table is recreated fresh; within the
# MERGE, facility_key and _first_build_id stay insert-only for subsequent builds.
id_map_df.createOrReplaceTempView("id_map_candidate")

spark.sql("""
CREATE OR REPLACE TABLE public_health.silver.facilities_id_map_silver (
    _source_key           STRING COMMENT 'which source the identifier comes from',
    source_record_id      STRING COMMENT 'nhfr_id for NHFR, otherwise licence no. or bronze row identity',
    source_id_raw         STRING,
    source_id_norm        STRING,
    code_validated        BOOLEAN,
    facility_key          STRING COMMENT 'surrogate uuid(), assigned once at first sight, never re-derived',
    match_method          STRING COMMENT 'nhfr_id / name_municipality / manual / new',
    match_status          STRING COMMENT 'matched / new_facility / ambiguous / candidate / service_of_existing_facility',
    facility_name         STRING,
    municipality_name     STRING,
    nhfr_id               BIGINT,
    disposition           STRING COMMENT 'working disposition from matching; full input accounting in quality.row_disposition',
    _first_build_id       STRING COMMENT 'build that first mapped the identifier (insert-only)',
    _silver_build_id      STRING COMMENT 'last build that read this identifier',
    _source_file          STRING,
    _source_sheet         STRING COMMENT 'null for CSV sources (NHFR)',
    _source_row_num       INT,
    _as_of_date           DATE
)
USING DELTA
COMMENT 'Stable facility identity map. Grain: 1 row per source identifier. Key: (_source_key, source_record_id). facility_key insert-only; MERGE history via build ids (plan 2.6, 5.2).'
""")

print(f"Table recreated with v1.5 schema: {TARGET_TABLE}")

In [0]:
# MERGE (plan 1, table 1): insert new source identifiers, update _silver_build_id for existing
# facility_key is insert-only (never updated once assigned); _first_build_id likewise

spark.sql("""
MERGE INTO public_health.silver.facilities_id_map_silver AS target
USING id_map_candidate AS source
ON  target._source_key = source._source_key
AND target.source_record_id = source.source_record_id

WHEN MATCHED THEN UPDATE SET
    -- v1.5 (plan 2.6, 5.3): build ids, not timestamps
    _silver_build_id = source._silver_build_id,
    match_method    = source.match_method,
    match_status    = source.match_status,
    disposition     = source.disposition,
    facility_name   = source.facility_name,
    municipality_name = source.municipality_name,
    nhfr_id         = source.nhfr_id,
    code_validated  = source.code_validated,
    source_id_raw   = source.source_id_raw,
    source_id_norm  = source.source_id_norm

WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE complete")

In [0]:
# v1.5 build sequence, step 4: record success (after the MERGE).
# build_log + build_inputs + row_disposition + quarantine appended only now.
written_count = spark.table(TARGET_TABLE).count()
assert written_count == EXPECTED_TOTAL_ROWS, f"Expected {EXPECTED_TOTAL_ROWS} id map rows, got {written_count}"
target_version = latest_delta_version(TARGET_TABLE)
build_finished_at = datetime.now(timezone.utc)

# Idempotent: clear any previous records for this build id (safe re-run)
for _t, _c in [(BUILD_LOG_TABLE, "build_id"), (BUILD_INPUTS_TABLE, "build_id"),
               (ROW_DISPOSITION_TABLE, "_silver_build_id"), (QUARANTINE_TABLE, "validation_id")]:
    spark.sql(f"DELETE FROM {_t} WHERE {_c} = '{SILVER_BUILD_ID}'")

# build_log: one row for this build
spark.createDataFrame(
    [(SILVER_BUILD_ID, "silver", TARGET_TABLE, BUILD_STARTED_AT, build_finished_at,
      "succeeded", int(written_count), int(target_version), None, None, None)],
    schema=spark.table(BUILD_LOG_TABLE).schema,
).write.format("delta").mode("append").saveAsTable(BUILD_LOG_TABLE)

# build_inputs: one row per file/sheet read (NHFR CSV + HFSRB Government/Private)
inputs_payload = [(
    SILVER_BUILD_ID, TARGET_TABLE, "bronze_source", input_table, int(version),
    source_key, file, sheet, batch_id, checksum, ingested_at, int(rows),
) for (input_table, version, source_key, file, sheet, batch_id, checksum, ingested_at, rows) in captured_inputs_full]
spark.createDataFrame(inputs_payload, schema=spark.table(BUILD_INPUTS_TABLE).schema) \
    .write.format("delta").mode("append").saveAsTable(BUILD_INPUTS_TABLE)

# row_disposition: one disposition per bronze row read (plan 6.1)
# loaded: NHFR rows + HFSRB rows that start a facility; merged_into: HFSRB rows
# joined to an NHFR facility (target_key = facility_key); quarantined: ambiguous rows
# One lookup row per input; the workbook sheets share a load, so join on
# (_source_key, _source_file, sheet null-safe) — a row can never double-match
batch_lookup = (
    spark.createDataFrame(
        [(source_key, file, sheet, batch_id, checksum)
         for (_, _, source_key, file, sheet, batch_id, checksum, ingested_at, rows) in captured_inputs_full],
        ["lk_source_key", "lk_source_file", "lk_source_sheet", "batch_id", "content_checksum"],
    )
    .dropDuplicates(["lk_source_key", "lk_source_file", "lk_source_sheet", "batch_id"])
)
dispositions = (
    id_map_df
    .join(
        batch_lookup,
        (F.col("_source_key") == F.col("lk_source_key"))
        & (F.col("_source_file") == F.col("lk_source_file"))
        & (F.col("_source_sheet").eqNullSafe(F.col("lk_source_sheet"))),
        "left",
    )
    .select(
        F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
        F.col("_source_key"),
        F.col("_source_file"),
        F.col("_source_sheet"),
        F.col("_source_row_num"),
        F.col("batch_id").alias("_load_batch_id"),
        F.col("content_checksum").alias("_content_checksum"),
        F.lit(TARGET_TABLE).alias("target_table"),
        F.when(F.col("match_status") == "matched", F.col("facility_key"))
         .otherwise(F.col("source_record_id")).alias("target_key"),
        F.when(F.col("match_status") == "matched", F.lit("merged_into"))
         .when(F.col("match_status") == "ambiguous", F.lit("quarantined"))
         .otherwise(F.lit("loaded")).alias("disposition"),
        F.when(F.col("match_status") == "ambiguous", F.lit("ambiguous_nhfr_code"))
         .otherwise(F.lit(None)).cast("string").alias("reason"),
    )
)
# Plan 6.1: exactly one disposition per bronze row read (no gaps, no doubles)
disp_count = dispositions.count()
assert disp_count == EXPECTED_TOTAL_ROWS, \
    f"Expected {EXPECTED_TOTAL_ROWS} dispositions, got {disp_count} — join fan-out check"
dispositions.write.format("delta").mode("append").saveAsTable(ROW_DISPOSITION_TABLE)

# Quarantine (plan 2.4): ambiguous nhfr_code rows, kept as JSON for review
ambiguous_rows = hfsrb_resolved.filter(F.col("dup_resolution_type") == "ambiguous").collect()
if ambiguous_rows:
    quarantine_payload = [(
        None, SILVER_BUILD_ID, "silver", TARGET_TABLE, HFSRB_BRONZE, None,
        "ambiguous_nhfr_code",
        json.dumps({"nhfr_id": str(r["nhfr_id"]), "facility_name": r["facility_name"],
                    "municipality_name": r["municipality_name"], "source_row_num": r["source_row_num"]}),
        json.dumps({k: str(v) for k, v in r.asDict().items()}),
        build_finished_at,
    ) for r in ambiguous_rows]
    spark.createDataFrame(quarantine_payload, schema=spark.table(QUARANTINE_TABLE).schema) \
        .write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)

print(f"Build records appended — build {SILVER_BUILD_ID}")
print(f"  build_log      : 1 row (succeeded, target Delta version {target_version})")
print(f"  build_inputs   : {len(inputs_payload)} inputs (NHFR CSV + HFSRB Government/Private)")
print(f"  row_disposition: {disp_count} rows")
print(f"  quarantine     : {len(ambiguous_rows)} ambiguous nhfr_code rows")

In [0]:
# Validation + summary
result = spark.sql("""
SELECT 
    _source_key,
    match_status,
    disposition,
    count(*) as row_count
FROM public_health.silver.facilities_id_map_silver
GROUP BY _source_key, match_status, disposition
ORDER BY _source_key, match_status, disposition
""")
display(result)

# Key validations
total = spark.table(TARGET_TABLE).count()
unique_keys = spark.table(TARGET_TABLE).select("facility_key").distinct().count()
unique_source_ids = spark.table(TARGET_TABLE).select("_source_key", "source_record_id").distinct().count()

print(f"\nTotal rows: {total}")
print(f"Unique facility_keys: {unique_keys}")
print(f"Unique source identifiers: {unique_source_ids}")

if total != unique_source_ids:
    dup_count = total - unique_source_ids
    print(f"WARNING: {dup_count} duplicate source identifiers found ({total} rows, {unique_source_ids} unique)")
    # Show the duplicates
    dup_source_ids = (
        spark.table(TARGET_TABLE)
        .groupBy("_source_key", "source_record_id")
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
        .orderBy(F.desc("row_count"))
    )
    display(dup_source_ids.limit(20))
else:
    print("Validation passed: no duplicate source identifiers")

# Show sample matched and unmatched rows
print("\nSample matched HFSRB rows:")
spark.sql("""
SELECT source_id_raw, facility_name, municipality_name, match_method, match_status, disposition
FROM public_health.silver.facilities_id_map_silver
WHERE _source_key = 'doh_hfsrb_hospitals' AND match_status = 'matched'
LIMIT 10
""").show(truncate=False)

print("Sample new facility HFSRB rows:")
spark.sql("""
SELECT source_id_raw, facility_name, municipality_name, match_method, match_status, disposition
FROM public_health.silver.facilities_id_map_silver
WHERE _source_key = 'doh_hfsrb_hospitals' AND match_status = 'new_facility'
LIMIT 10
""").show(truncate=False)

print("\nID Map build complete.")